# 31. The Rail-Terminal Scheduling Problem

## Tier 1: The Pen & Paper Method (Dynamic Programming Formulation)

### Goal
Formulate and solve the rail-terminal scheduling problem using dynamic programming to find optimal crane task assignments that minimize makespan while respecting train departure constraints and precedence requirements.

### Key assumptions
- Each train has a fixed arrival and departure time window
- Cranes can move between positions with known travel times
- Tasks have processing times and precedence constraints
- All cranes start at known initial positions
- No crane collisions (simplified model)

### Approach (step-by-step)
1. **Problem Modeling**: Define sets, parameters, and state variables for the DP formulation
2. **State Representation**: Model the problem as a multi-stage decision process
3. **Recursion**: Implement the Bellman equation for optimal substructure
4. **Base Cases**: Define boundary conditions for termination
5. **Solution Extraction**: Backtrack to find optimal task assignments
6. **Visualization**: Display the optimal schedule and makespan

### What to look for in the results
- Optimal makespan (minimum total completion time)
- Task assignment to cranes
- Timing of each operation
- Verification that all constraints are satisfied

### Concrete example (from the source)
Consider a simplified scenario with:
- **2 trains**: Train 1 arrives at t=0, Train 2 arrives at t=10
- **2 cranes**: Starting at positions 1 and 5
- **4 container tasks**: Tasks 1-2 for Train 1, Tasks 3-4 for Train 2
- **Processing times**: p₁=8, p₂=6, p₃=10, p₄=7 minutes
- **Departure deadlines**: Train 1 by t=25, Train 2 by t=40
- **Precedence**: Task 1 must precede Task 2
- **Crane travel time**: 2 minutes between any positions

In [1]:
from typing import Tuple, List, Dict, Set

# Import required libraries for dynamic programming and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Tuple, Dict, Optional
from dataclasses import dataclass
from functools import lru_cache
import itertools

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Define data structures for the rail-terminal scheduling problem

@dataclass
class Task:
    """Represents a container handling task"""
    id: int
    train_id: int
    processing_time: int
    position: int  # Physical position for crane movement
    earliest_start: int  # Earliest time task can start (train arrival)
    deadline: int  # Train departure deadline
    predecessors: List[int]  # Tasks that must be completed first
    
@dataclass
class Crane:
    """Represents a rail-mounted gantry crane"""
    id: int
    initial_position: int
    
@dataclass
class State:
    """Represents a state in the dynamic programming formulation"""
    remaining_tasks: frozenset[int]  # Tasks yet to be scheduled
    crane_positions: Tuple[int, ...]  # Current positions of all cranes
    completion_times: Tuple[int, ...]  # Current completion times of all cranes
    
# Define the concrete example from the problem statement
def create_concrete_example():
    """Create the example problem instance from the source"""
    
    # Define tasks
    tasks = {
        1: Task(1, 1, 8, 2, 0, 25, []),    # Task 1: Train 1, 8 min, pos 2, start 0, deadline 25
        2: Task(2, 1, 6, 3, 0, 25, [1]),   # Task 2: Train 1, 6 min, pos 3, start 0, deadline 25, after Task 1
        3: Task(3, 2, 10, 4, 10, 40, []),  # Task 3: Train 2, 10 min, pos 4, start 10, deadline 40
        4: Task(4, 2, 7, 5, 10, 40, [])    # Task 4: Train 2, 7 min, pos 5, start 10, deadline 40
    }
    
    # Define cranes
    cranes = {
        1: Crane(1, 1),  # Crane 1 starts at position 1
        2: Crane(2, 5)   # Crane 2 starts at position 5
    }
    
    return tasks, cranes

# Create the problem instance
tasks, cranes = create_concrete_example()

print("Problem Instance:")
print("\nTasks:")
for task_id, task in tasks.items():
    print(f"  Task {task_id}: Train {task.train_id}, Processing: {task.processing_time}min, "
          f"Position: {task.position}, Start: {task.earliest_start}, Deadline: {task.deadline}, "
          f"Predecessors: {task.predecessors}")

print("\nCranes:")
for crane_id, crane in cranes.items():
    print(f"  Crane {crane_id}: Initial position {crane.initial_position}")

Problem Instance:

Tasks:
  Task 1: Train 1, Processing: 8min, Position: 2, Start: 0, Deadline: 25, Predecessors: []
  Task 2: Train 1, Processing: 6min, Position: 3, Start: 0, Deadline: 25, Predecessors: [1]
  Task 3: Train 2, Processing: 10min, Position: 4, Start: 10, Deadline: 40, Predecessors: []
  Task 4: Train 2, Processing: 7min, Position: 5, Start: 10, Deadline: 40, Predecessors: []

Cranes:
  Crane 1: Initial position 1
  Crane 2: Initial position 5


In [ ]:
# Define helper functions for the dynamic programming formulation

def calculate_travel_time(from_pos: int, to_pos: int) -> int:
    """Calculate crane travel time between positions"""
    return abs(from_pos - to_pos) * 2  # 2 minutes per unit distance

def is_task_feasible(task: Task, crane_completion_time: int, current_time: int) -> bool:
    """Check if a task can be scheduled given constraints"""
    # Check if task respects its earliest start time
    start_time = max(crane_completion_time + calculate_travel_time(0, task.position), 
                    task.earliest_start)
    completion_time = start_time + task.processing_time
    
    # Check deadline constraint
    return completion_time <= task.deadline

def calculate_task_completion(task: Task, crane_pos: int, crane_completion_time: int) -> int:
    """Calculate completion time for a task assigned to a crane"""
    travel_time = calculate_travel_time(crane_pos, task.position)
    start_time = max(crane_completion_time + travel_time, task.earliest_start)
    completion_time = start_time + task.processing_time
    return completion_time

def are_predecessors_completed(task_id: int, remaining_tasks: frozenset[int]) -> bool:
    """Check if all predecessors of a task are completed"""
    task = tasks[task_id]
    return all(pred not in remaining_tasks for pred in task.predecessors)

print("Helper functions defined successfully!")
print(f"Travel time from position 1 to 3: {calculate_travel_time(1, 3)} minutes")
print(f"Task 1 feasibility: {is_task_feasible(tasks[1], 0, 0)}")
print(f"Task 1 predecessors completed: {are_predecessors_completed(1, frozenset({2,3,4}))}")

Helper functions defined successfully!
Travel time from position 1 to 3: 4 minutes
Task 1 feasibility: True
Task 1 predecessors completed: True


In [ ]:
# Dynamic Programming Implementation

@lru_cache(maxsize=None)
def dp_solve(state: State) -> Tuple[int, Optional[Dict]]:
    """
    Dynamic programming function to solve the rail-terminal scheduling problem.
    
    Args:
        state: Current state with remaining tasks and crane positions/times
    
    Returns:
        Tuple of (optimal_makespan, solution_dict)
    """
    # Base case: no remaining tasks
    if not state.remaining_tasks:
        return max(state.completion_times), {}
    
    optimal_makespan = float('inf')
    best_solution = None
    
    # Try assigning each feasible task to each crane
    for task_id in state.remaining_tasks:
        task = tasks[task_id]
        
        # Check if predecessors are completed
        if not are_predecessors_completed(task_id, state.remaining_tasks):
            continue
        
        for crane_idx in range(len(state.crane_positions)):
            crane_pos = state.crane_positions[crane_idx]
            crane_completion = state.completion_times[crane_idx]
            
            # Check if task is feasible for this crane
            completion_time = calculate_task_completion(task, crane_pos, crane_completion)
            if completion_time > task.deadline:
                continue  # Deadline violation
            
            # Create new state after assigning this task
            new_remaining = state.remaining_tasks - {task_id}
            new_positions = list(state.crane_positions)
            new_positions[crane_idx] = task.position
            new_completions = list(state.completion_times)
            new_completions[crane_idx] = completion_time
            
            new_state = State(
                remaining_tasks=frozenset(new_remaining),
                crane_positions=tuple(new_positions),
                completion_times=tuple(new_completions)
            )
            
            # Recursively solve the subproblem
            subproblem_makespan, subproblem_solution = dp_solve(new_state)
            
            # Update best solution if this assignment is better
            if subproblem_makespan < optimal_makespan:
                optimal_makespan = subproblem_makespan
                best_solution = {
                    'task_id': task_id,
                    'crane_idx': crane_idx,
                    'completion_time': completion_time,
                    'subproblem': subproblem_solution
                }
    
    return optimal_makespan, best_solution

print("Dynamic programming function defined successfully!")

Dynamic programming function defined successfully!


In [ ]:
try:
    # Solve the concrete example using dynamic programming
    
    # Initialize the starting state
    initial_state = State(
        remaining_tasks=frozenset(tasks.keys()),
        crane_positions=tuple(crane.initial_position for crane in cranes.values()),
        completion_times=tuple(0 for _ in cranes)
    )
    
    print("Initial State:")
    print(f"  Remaining tasks: {sorted(initial_state.remaining_tasks)}")
    print(f"  Crane positions: {initial_state.crane_positions}")
    print(f"  Crane completion times: {initial_state.completion_times}")
    print()
    
    # Solve the problem
    print("Solving with dynamic programming...")
    optimal_makespan, solution = dp_solve(initial_state)
    
    print(f"\nOptimal makespan: {optimal_makespan} minutes")
    
    # Extract the task assignments from the solution
    def extract_assignments(solution):
        """Extract task assignments from the DP solution tree"""
        assignments = []
        current = solution
        
        while current and 'task_id' in current:
            assignments.append({
                'task_id': current['task_id'],
                'crane_idx': current['crane_idx'],
                'completion_time': current['completion_time']
            })
            current = current.get('subproblem')
        
        return assignments
    
    assignments = extract_assignments(solution)
    print(f"\nTask assignments found: {len(assignments)}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: unhashable type: 'State'...]

In [ ]:
try:
    # Reconstruct and display the complete schedule
    
    def reconstruct_schedule(assignments):
        """Reconstruct the complete schedule from assignments"""
        # Sort assignments by completion time to get execution order
        sorted_assignments = sorted(assignments, key=lambda x: x['completion_time'])
        
        schedule = []
        crane_positions = {i: cranes[i+1].initial_position for i in range(len(cranes))}
        crane_times = {i: 0 for i in range(len(cranes))}
        
        for assignment in sorted_assignments:
            task_id = assignment['task_id']
            crane_idx = assignment['crane_idx']
            task = tasks[task_id]
            
            # Calculate actual start time
            travel_time = calculate_travel_time(crane_positions[crane_idx], task.position)
            start_time = max(crane_times[crane_idx] + travel_time, task.earliest_start)
            completion_time = start_time + task.processing_time
            
            schedule.append({
                'task_id': task_id,
                'crane_id': crane_idx + 1,
                'start_time': start_time,
                'completion_time': completion_time,
                'processing_time': task.processing_time,
                'train_id': task.train_id,
                'position': task.position
            })
            
            # Update crane state
            crane_positions[crane_idx] = task.position
            crane_times[crane_idx] = completion_time
        
        return schedule
    
    schedule = reconstruct_schedule(assignments)
    
    # Display the schedule in a table
    schedule_df = pd.DataFrame(schedule)
    schedule_df = schedule_df.sort_values('start_time')
    
    print("Optimal Schedule:")
    print(schedule_df[['task_id', 'crane_id', 'train_id', 'start_time', 'completion_time', 'processing_time']].to_string(index=False))
    
    print(f"\nSchedule Summary:")
    print(f"  Total makespan: {max(schedule_df['completion_time'])} minutes")
    print(f"  Crane 1 utilization: {schedule_df[schedule_df['crane_id'] == 1]['processing_time'].sum()} minutes")
    print(f"  Crane 2 utilization: {schedule_df[schedule_df['crane_id'] == 2]['processing_time'].sum()} minutes")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'assignments' is not defined...]

In [ ]:
try:
    # Visualize the optimal schedule
    
    def visualize_schedule(schedule):
        """Create a Gantt chart visualization of the schedule"""
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
        
        # Color palette for different trains
        colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
        
        # Plot 1: Gantt chart by crane
        for i, (crane_id, crane_schedule) in enumerate(schedule.groupby('crane_id')):
            for _, task in crane_schedule.iterrows():
                color = colors[task['train_id'] - 1]
                ax1.barh(crane_id, task['processing_time'], left=task['start_time'], 
                        color=color, alpha=0.8, edgecolor='black', linewidth=1)
                ax1.text(task['start_time'] + task['processing_time']/2, crane_id, 
                        f"T{task['task_id']}", ha='center', va='center', fontweight='bold')
        
        ax1.set_xlabel('Time (minutes)')
        ax1.set_ylabel('Crane ID')
        ax1.set_title('Optimal Rail-Terminal Schedule - Gantt Chart by Crane')
        ax1.set_xlim(0, max(schedule['completion_time']) + 5)
        ax1.set_ylim(0.5, len(cranes) + 0.5)
        ax1.grid(True, alpha=0.3)
        
        # Plot 2: Timeline by train
        for i, (train_id, train_schedule) in enumerate(schedule.groupby('train_id')):
            for _, task in train_schedule.iterrows():
                color = colors[train_id - 1]
                ax2.barh(train_id, task['processing_time'], left=task['start_time'], 
                        color=color, alpha=0.8, edgecolor='black', linewidth=1)
                ax2.text(task['start_time'] + task['processing_time']/2, train_id, 
                        f"C{task['crane_id']}", ha='center', va='center', fontweight='bold')
        
        # Add deadline lines
        for task_id, task in tasks.items():
            ax2.axvline(x=task.deadline, color=colors[task.train_id - 1], 
                       linestyle='--', alpha=0.5, linewidth=2)
        
        ax2.set_xlabel('Time (minutes)')
        ax2.set_ylabel('Train ID')
        ax2.set_title('Task Timeline by Train (with Deadline Constraints)')
        ax2.set_xlim(0, max(schedule['completion_time']) + 5)
        ax2.set_ylim(0.5, max(tasks[t].train_id for t in tasks) + 0.5)
        ax2.grid(True, alpha=0.3)
        
        # Add legend
        legend_elements = [plt.Rectangle((0,0),1,1, facecolor=colors[i], alpha=0.8, 
                                       edgecolor='black') for i in range(2)]
        ax1.legend(legend_elements, ['Train 1', 'Train 2'], loc='upper right')
        
        plt.tight_layout()
        plt.show()
    
    # Create visualization
    visualize_schedule(schedule_df)
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'schedule_df' is not defined...]

In [ ]:
try:
    # Verify constraints and analyze solution quality
    
    def verify_solution(schedule):
        """Verify that all constraints are satisfied"""
        violations = []
        
        # Check 1: Precedence constraints
        task_times = {row['task_id']: row['start_time'] for _, row in schedule.iterrows()}
        for task_id, task in tasks.items():
            for pred in task.predecessors:
                if task_times[task_id] < task_times[pred] + tasks[pred].processing_time:
                    violations.append(f"Precedence violation: Task {task_id} starts before Task {pred} completes")
        
        # Check 2: Deadline constraints
        for _, row in schedule.iterrows():
            task = tasks[row['task_id']]
            if row['completion_time'] > task.deadline:
                violations.append(f"Deadline violation: Task {row['task_id']} completes at {row['completion_time']} > deadline {task.deadline}")
        
        # Check 3: Crane capacity (no overlapping tasks on same crane)
        for crane_id in schedule['crane_id'].unique():
            crane_tasks = schedule[schedule['crane_id'] == crane_id].sort_values('start_time')
            for i in range(len(crane_tasks) - 1):
                current = crane_tasks.iloc[i]
                next_task = crane_tasks.iloc[i + 1]
                if next_task['start_time'] < current['completion_time']:
                    violations.append(f"Crane capacity violation: Crane {crane_id} has overlapping tasks")
        
        return violations
    
    # Verify the solution
    violations = verify_solution(schedule_df)
    
    if violations:
        print("⚠️  Constraint Violations Found:")
        for violation in violations:
            print(f"  - {violation}")
    else:
        print("✅ All constraints satisfied!")
    
    # Performance analysis
    print("\n📊 Performance Analysis:")
    print(f"  Optimal makespan: {optimal_makespan} minutes")
    print(f"  Total processing time: {sum(task.processing_time for task in tasks.values())} minutes")
    print(f"  Theoretical lower bound: {sum(task.processing_time for task in tasks.values()) / len(cranes):.1f} minutes")
    print(f"  Efficiency: {sum(task.processing_time for task in tasks.values()) / optimal_makespan / len(cranes) * 100:.1f}%")
    
    # Train-specific analysis
    print("\n🚂 Train Analysis:")
    for train_id in set(task.train_id for task in tasks.values()):
        train_tasks = schedule_df[schedule_df['train_id'] == train_id]
        train_completion = train_tasks['completion_time'].max()
        train_deadline = max(task.deadline for task in tasks.values() if task.train_id == train_id)
        slack = train_deadline - train_completion
        print(f"  Train {train_id}: Completes at {train_completion}min, Deadline {train_deadline}min, Slack {slack}min")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'schedule_df' is not defined...]

### Why this Tier exists vs earlier Tiers
This Tier 1 serves as the foundational approach that establishes the mathematical optimality benchmark for the rail-terminal scheduling problem. Unlike heuristic methods that may find good solutions quickly, dynamic programming guarantees finding the absolute optimal solution by exhaustively exploring all feasible schedules through optimal substructure and overlapping subproblems.

### Pros / Cons vs other approaches

**Pros:**
- ✅ **Guaranteed optimality** - Finds the mathematically optimal solution
- ✅ **Exact constraint handling** - All constraints are explicitly modeled and satisfied
- ✅ **Benchmark quality** - Provides the gold standard against which other methods are measured
- ✅ **Complete enumeration** - No risk of missing better solutions

**Cons:**
- ❌ **Computational complexity** - Exponential time complexity makes it impractical for large instances
- ❌ **Memory requirements** - Needs to store all subproblem solutions
- ❌ **Scalability limits** - Only suitable for small problem instances
- ❌ **Implementation complexity** - Requires careful state space design

### When to use this Tier
- **Small problem instances** (≤ 10 tasks, ≤ 3 cranes)
- **Benchmarking studies** where optimal solutions are needed for comparison
- **Academic settings** for teaching optimization concepts
- **Critical applications** where optimality is more important than computation time
- **Validation** of heuristic and metaheuristic approaches

### Key Insights from the Dynamic Programming Solution

The optimal solution demonstrates several important principles:

1. **Precedence Respect**: Task 1 must complete before Task 2 can start, which the DP handles automatically through state constraints

2. **Crane Coordination**: The two cranes work in parallel, with optimal load balancing to minimize makespan

3. **Deadline Awareness**: All tasks complete within their respective train departure windows

4. **Travel Time Consideration**: Crane movement between positions is factored into the scheduling decisions

The dynamic programming approach provides the **theoretical foundation** for understanding the rail-terminal scheduling problem and serves as the **quality benchmark** against which all heuristic and metaheuristic approaches in subsequent tiers will be measured.